# MCP Agent 动手实践
## 2026-05-07 | 第1周周四 | MCP 协议改造

> **学习目标**: 将昨天的 Function Calling Agent 改造为 MCP (Model Context Protocol) 架构，理解工具的标准化暴露与发现机制

**你将学到：**
- MCP 协议的 Client-Server 架构（Tools / Resources / Prompts 三大原语）
- 如何将工具函数包装为 MCP Server
- MCP Tool Schema 与 OpenAI Function Calling 格式的转换
- MCP Client 如何发现和调用远程工具
- Agent 循环在 MCP 架构下的实现

**架构对比：**
```
昨天:  用户 → Agent Loop → LLM → tool_calls → 本地执行工具 → 回传
今天:  用户 → Agent Loop → LLM → tool_calls → MCP Client → MCP Server → 执行 → 回传
```

---

## 新手导读：MCP 是把工具从 Agent 里拆出去的标准接口

如果昨天的 Function Calling 是“Agent 自己带工具箱”，今天的 MCP 更像“Agent 通过标准插口连接外部工具箱”。

核心区别：

- Function Calling：工具 schema 和执行函数通常写在 Agent 进程里。
- MCP：工具由 MCP Server 暴露，Agent 作为 MCP Client 去发现工具、调用工具。
- LLM 仍然看到 OpenAI/DeepSeek 风格的 tools；只是这些 tools 背后由 MCP Client 转发给 Server。

阅读顺序建议：

1. 先看工具函数本身：它们和昨天类似，说明 MCP 并不强迫你改变业务逻辑。
2. 再看 MCP Server：它负责“注册工具”和“响应调用”。
3. 最后看 MCP Client：它把 server 发现到的工具转换成 LLM 可理解的 schema。

常见卡点：

- MCP 不等于模型协议，它是工具协议；模型仍然通过 chat completions 产生 tool calls。
- `stdio` 模式不是网络服务，它通过标准输入输出通信，适合本地工具和开发调试。
- Server 和 Client 的职责要分清：Server 不负责问 LLM，Client/Agent 才负责模型循环。


## 1. 环境准备

初始化 DeepSeek 异步客户端 + MCP 依赖。

> MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk

In [ ]:
import asyncio
import json
import math
import os
import sys

from dotenv import load_dotenv
from openai import AsyncOpenAI
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession
from mcp.types import Tool, TextContent

load_dotenv()

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

# 使用异步客户端：MCP 的 Client-Server 通信是 async 的，
# 如果 LLM 调用也是同步的会阻塞事件循环，所以这里配套用 AsyncOpenAI
llm_client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url="https://api.deepseek.com",
)

print("环境就绪!")
print(f"  MCP SDK 可用")
print(f"  AsyncOpenAI 客户端已初始化")

## 2. MCP 协议概览（八股题 12）

### MCP 是什么？

MCP (Model Context Protocol) 是 Anthropic 提出的**开放协议**，定义了 LLM 应用如何与外部工具、资源进行标准化的 Client-Server 交互。

### 三大核心原语

| 原语 | 用途 | 类比 |
|------|------|------|
| **Tools** | LLM 可调用的函数 | API 端点 |
| **Resources** | 服务端暴露的数据 | REST 资源 |
| **Prompts** | 预定义的提示词模板 | 快捷指令 |

### 传输层

| 方式 | 场景 | 本文件使用 |
|------|------|-----------|
| **stdio** | 本地进程通信 | ✅ |
| **HTTP/SSE** | 远程服务通信 | — |

### 为什么需要 MCP？

```
没有 MCP：
  每个 Agent 项目都硬编码工具 → 工具不可复用 → 实现与逻辑耦合

有了 MCP：
  MCP Server（工具提供方） ←→ MCP Client（工具消费方）
  同一套工具可以被 Claude Desktop、Cursor、自定义 Agent 等任意客户端调用
```

### MCP Client-Server 交互流程

```
┌──────────┐  ① stdio 启动子进程    ┌──────────┐
│  Client  │ ────────────────────→ │  Server  │
│          │  ② initialize() 握手   │          │
│          │ ←──────────────────── │          │
│          │  ③ list_tools() 发现   │          │
│          │ ←──── 返回 Tool[] ──── │          │
│          │  ④ call_tool() 调用   │          │
│          │ ────────────────────→ │          │
│          │ ←── 返回 CallToolResult│          │
└──────────┘                       └──────────┘
```

### MCP vs Function Calling（八股题 13）

这两个概念经常被混淆，但它们解决的是不同层面的问题：

| 维度 | Function Calling | MCP |
|------|-----------------|-----|
| **是什么** | LLM 的**能力** | 工具的**标准化协议** |
| **解决的问题** | LLM 如何表达"我想调函数" | 工具如何被发现、描述、调用 |
| **定义者** | LLM 厂商 (OpenAI/Anthropic) | 开放协议 (Anthropic 主导) |
| **格式** | 各厂商不同 | 统一 JSON Schema |
| **关系** | 消费端 | 供应链端 |

**一句话总结**: MCP 负责"工具有哪些、怎么调"，Function Calling 负责"LLM 调用意图的表达"。二者互补。

## 3. 工具实现（真正干活的代码）

> **关键认知**: 这些函数与昨天的实现**完全一致**！MCP 改造不涉及工具逻辑本身——
> 只是给它们包了一层标准化的**MCP 外壳**，让任何 MCP 客户端都能发现和调用。

In [ ]:
# ── 天气查询实现（与昨天完全一致）──
# MCP 改造的关键洞察：工具的业务逻辑代码完全不用动，
# 只是在外面套了一层 MCP "包装壳"，让它能被任何 MCP Client 调用
def get_weather(city: str, unit: str = "celsius") -> dict:
    """查询城市天气（当前为模拟数据）"""
    weather_db = {
        "北京":     {"temp_c": 22, "condition": "晴",     "humidity": 40, "wind": "北风 3级"},
        "上海":     {"temp_c": 25, "condition": "多云",   "humidity": 68, "wind": "东南风 2级"},
        "广州":     {"temp_c": 29, "condition": "雷阵雨", "humidity": 85, "wind": "南风 4级"},
        "深圳":     {"temp_c": 28, "condition": "阴",     "humidity": 78, "wind": "东风 3级"},
        "杭州":     {"temp_c": 24, "condition": "小雨",   "humidity": 72, "wind": "东北风 2级"},
        "成都":     {"temp_c": 21, "condition": "阴",     "humidity": 75, "wind": "无持续风向 1级"},
        "tokyo":    {"temp_c": 18, "condition": "晴",     "humidity": 50, "wind": "北风 2级"},
        "london":   {"temp_c": 13, "condition": "小雨",   "humidity": 80, "wind": "西风 5级"},
        "new york": {"temp_c": 16, "condition": "多云",   "humidity": 55, "wind": "西南风 4级"},
        "sydney":   {"temp_c": 20, "condition": "晴",     "humidity": 45, "wind": "东风 3级"},
    }
    key = city.strip().lower()
    data = weather_db.get(key, {"temp_c": 20, "condition": "暂无数据", "humidity": 60, "wind": "未知"})
    temp = data["temp_c"]
    unit_label = "°C"
    if unit == "fahrenheit":
        temp = round(temp * 9/5 + 32, 1)
        unit_label = "°F"
    return {
        "city": city, "temperature": temp, "unit": unit_label,
        "condition": data["condition"], "humidity": f"{data['humidity']}%", "wind": data["wind"]
    }


# ── 计算器实现（与昨天完全一致）──
def calculate(expression: str) -> dict:
    """安全执行数学表达式（受限 eval 沙箱）"""
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "min": min, "max": max, "pow": pow})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)
        return {"expression": expression, "result": result, "error": None}
    except Exception as e:
        return {"expression": expression, "result": None, "error": str(e)}


TOOL_EXECUTORS = {"get_weather": get_weather, "calculate": calculate}


def execute_tool(name: str, args: dict) -> str:
    """工具统一调度入口——与昨天完全一致"""
    func = TOOL_EXECUTORS.get(name)
    if func is None:
        return json.dumps({"error": f"未知工具: {name}"}, ensure_ascii=False)
    try:
        return json.dumps(func(**args), ensure_ascii=False)
    except Exception as e:
        return json.dumps({"error": str(e)}, ensure_ascii=False)


# 快速验证
print(">>> execute_tool('get_weather', {'city': '北京'}):")
print("   ", execute_tool("get_weather", {"city": "北京"}))
print("\n>>> execute_tool('calculate', {'expression': 'sqrt(256) + 2**10'}):")
print("   ", execute_tool("calculate", {"expression": "sqrt(256) + 2**10"}))
print("\n工具函数就绪！")

## 4. MCP Server —— 工具的标准化"包装壳"

这是今天与昨天的**核心区别**：
- 昨天：工具 Schema 硬编码为 `TOOLS = [WEATHER_TOOL, CALCULATOR_TOOL]`
- 今天：工具 Schema 由 MCP Server 动态提供，Client 通过 `list_tools()` 发现

### MCP Server 的两个核心方法

| 方法 | 装饰器 | 作用 |
|------|--------|------|
| `handle_list_tools()` | `@server.list_tools()` | 返回可用工具列表（Tool Schema） |
| `handle_call_tool()` | `@server.call_tool()` | 接收调用请求，执行并返回结果 |

> **面试要点**: MCP Tool 的 `inputSchema` 就是标准 JSON Schema，
> 与 OpenAI Function Calling 的 `parameters` 字段**完全兼容**。
> 这意味着 MCP Tool Schema 可以直接映射为任何 LLM 的工具格式。

In [ ]:
# ── 创建 MCP Server 实例 ──
# Server 的 name 只是标识符，Client 在握手时会看到它
server = Server("agent-tools")


@server.list_tools()
async def handle_list_tools() -> list[Tool]:
    """MCP 核心原语①：列出可用工具。

    每次 Client 调用 list_tools()，Server 返回完整的工具 Schema 列表。
    Client 拿到这个列表后，转换成 LLM 可用的 Function Calling 格式。
    这种"动态发现"机制让 Client 无需提前硬编码工具，Server 可随时增删工具。
    """
    return [
        Tool(
            name="get_weather",
            description=(
                "查询指定城市的实时天气信息。"
                "返回数据包含：温度、天气状况、湿度、风速。"
                "适用场景：用户询问某地天气怎么样、是否需要带伞等。"
            ),
            # inputSchema 和 OpenAI 的 parameters 字段完全兼容——都是标准 JSON Schema
            inputSchema={
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，支持中文或英文，例如：北京、Tokyo、London",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "温度单位，默认celsius（摄氏度）",
                    },
                },
                "required": ["city"],
            },
        ),
        Tool(
            name="calculate",
            description=(
                "执行数学表达式计算。"
                "支持：+-*/四则运算、**幂运算、三角函数(sin/cos/tan)、"
                "sqrt平方根、log/log10对数、abs绝对值。"
                "当用户需要精确数值计算时必须调用此工具，禁止心算。"
            ),
            inputSchema={
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式字符串，如 '(2+3)*4'、'sqrt(144)'",
                    }
                },
                "required": ["expression"],
            },
        ),
    ]


@server.call_tool()
async def handle_call_tool(name: str, arguments: dict) -> list[TextContent]:
    """MCP 核心原语②：执行工具调用。

    Client 发来工具名 + 参数 → Server 转发给真正的工具函数 → 返回 TextContent。
    TextContent 是 MCP 标准的返回类型，可以包含文本、图片等多种内容。
    这里直接复用 execute_tool，保证 MCP 版本和手写版本的业务逻辑完全一致。
    """
    result_json = execute_tool(name, arguments)
    return [TextContent(type="text", text=result_json)]


print("MCP Server 已定义")
print(f"  Server name: {server.name}")
print(f"  工具: get_weather, calculate")
print(f"  传输: stdio (JSON-RPC)")

## 5. MCP → OpenAI Tool Schema 转换

MCP Tool 与 OpenAI Function Calling Tool 的格式几乎一致，区别仅在外层包装：

```
MCP Tool:    { name, description, inputSchema }
OpenAI Tool: { type: "function", function: { name, description, parameters } }
```

`inputSchema` 就是标准 JSON Schema，直接映射到 `parameters` 字段即可。

> **这是理解 MCP 价值的关键**：MCP 定义了一次工具 Schema，Client 可以把它转成任何 LLM 需要的格式。OpenAI、Anthropic、Gemini 的格式不同，但 MCP Tool Schema 是统一的。

In [ ]:
def mcp_tools_to_openai(mcp_tools: list) -> list[dict]:
    """将 MCP Tool 列表转换为 OpenAI Function Calling 格式。

    MCP Tool 的 inputSchema 就是标准 JSON Schema，
    直接映射到 OpenAI 的 parameters 字段即可。
    这个转换层是 MCP 设计的关键价值：同一套工具 Schema 可以
    适配不同 LLM 的格式（OpenAI、Anthropic、Gemini 格式各不同）。
    """
    openai_tools = []
    for tool in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,   # MCP inputSchema = OpenAI parameters
            },
        })
    return openai_tools


# 演示：用 MCP Tool 对象测试转换
demo_tool = Tool(
    name="get_weather",
    description="查询指定城市的实时天气",
    inputSchema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"]
    }
)
openai_format = mcp_tools_to_openai([demo_tool])
print("MCP Tool → OpenAI 格式:")
print(json.dumps(openai_format[0], indent=2, ensure_ascii=False))

## 6. System Prompt

与昨天保持一致——System Prompt 定义 Agent 的行为边界，与 MCP 无关。

In [ ]:
SYSTEM_PROMPT = """你是一个具备工具调用能力的智能助手。你拥有以下工具：

1. get_weather — 查询任意城市的实时天气（温度、天气状况、湿度、风速）
2. calculate   — 执行数学表达式计算（支持四则运算、幂运算、三角函数等）

行为准则：
- 用户询问天气相关信息时，主动调用 get_weather
- 用户需要数值计算时，调用 calculate，禁止自行心算
- 收到工具返回结果后，用流畅的中文向用户转述
- 如果用户同时问了天气和计算，可以一次调用多个工具（并行调用）
- 保持回答简洁、信息密度高"""

# System Prompt 与 MCP 完全无关，这也是 MCP 架构的好处：
# 工具调用协议和模型指令互相解耦，各自独立演化
print("System Prompt 已加载，长度:", len(SYSTEM_PROMPT), "字符")

## 7. Agent 循环 —— MCP 架构版

与昨天的 Agent 循环核心逻辑**完全一致**，但工具调用链路不同：

```
昨天:  LLM 返回 tool_calls → execute_tool(name, args) → 本地执行 → 回传结果
今天:  LLM 返回 tool_calls → session.call_tool(name, args) → MCP Server 执行 → 回传结果
```

### 关键区别清单（面试重点）

| # | 区别点 | 昨天（纯 FC） | 今天（MCP） |
|---|--------|-------------|-----------|
| ① | **工具发现** | 硬编码 `TOOLS` 列表 | 从 Server 动态 `list_tools()` |
| ② | **工具执行** | 本地 `execute_tool()` | 通过 `session.call_tool()` 远程调用 |
| ③ | **格式转换** | 直接写 OpenAI 格式 | MCP Tool → 转换 → OpenAI 格式 |
| ④ | **结果提取** | 直接 JSON 字符串 | `result.content[0].text` |

> **核心认知**: Agent 循环的"思考→调用→反馈"逻辑不变，变的是调用链路的标准化程度。

In [ ]:
async def run_agent(
    session: ClientSession,
    llm_client: AsyncOpenAI,
    user_query: str,
    model: str = "deepseek-v4-flash",
    max_turns: int = 10,
    verbose: bool = True,
) -> str:
    """Agent 主循环 —— 通过 MCP Session 调用工具。

    与昨天 run_agent 的核心区别：
      昨天: result = execute_tool(name, args)          # 本地函数调用
      今天: result = await session.call_tool(name, args) # MCP 远程调用
    其他逻辑（消息历史管理、LLM 调用、判断 tool_calls）完全相同。
    """
    # ── 第0步：从 MCP Server 动态发现工具 ──
    # 这是 MCP 最核心的价值：Client 不硬编码工具，而是运行时从 Server 查询
    # 如果 Server 新增了工具，Client 无需修改代码就能用到
    tools_result = await session.list_tools()
    mcp_tools = tools_result.tools
    openai_tools = mcp_tools_to_openai(mcp_tools)

    if verbose:
        print(f"[MCP] 发现 {len(mcp_tools)} 个工具: "
              f"{', '.join(t.name for t in mcp_tools)}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for turn in range(1, max_turns + 1):
        response = await llm_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto",
            temperature=0.0,
        )

        msg = response.choices[0].message

        if msg.tool_calls:
            if verbose:
                names = [tc.function.name for tc in msg.tool_calls]
                print(f"\n[轮次 {turn}] 调用工具: {', '.join(names)}")

            messages.append(msg.model_dump())

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                if verbose:
                    print(f"  IN  {tool_name}({json.dumps(tool_args, ensure_ascii=False)})")

                # ── 关键区别：通过 MCP Session 远程调用工具 ──
                # session.call_tool 发送 JSON-RPC 请求给 Server 子进程
                # Server 执行完后通过 stdio 把结果送回
                result = await session.call_tool(tool_name, tool_args)
                # MCP 返回的是结构化 CallToolResult，文本在 content[0].text 里
                result_text = result.content[0].text

                if verbose:
                    print(f"  OUT {result_text}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_text,
                })

        else:
            if verbose:
                print(f"\n[轮次 {turn}] 最终回复 (文本)")
            return msg.content

    return "处理超时，请将问题拆分为更小的子问题。"


print("Agent 循环函数已定义（MCP 架构版）")

## 8. MCP 连接助手

定义一个 `ask()` 辅助函数——每次调用自动完成 MCP 连接 → 工具发现 → Agent 执行 → 清理。
每个测试 Cell 只需一行 `await ask("你的问题")`。

In [ ]:
import os as _os

async def ask(user_query: str, verbose: bool = True) -> str:
    """MCP Agent 一键查询助手。

    每次调用完整经历：
      1. 用 `uv run python agent.py serve` 启动 MCP Server 子进程
      2. 通过 stdio 建立 JSON-RPC 连接
      3. 握手初始化（exchanging capabilities）
      4. 发现工具 → 执行 Agent 循环 → 返回答案
      5. 自动清理 Session 和子进程

    为什么每次都新建连接？因为 Notebook 中重复运行 Cell 方便，
    生产系统应该复用长连接以减少握手开销。
    """
    server_params = StdioServerParameters(
        command="uv",
        args=["run", "python", "exercises/w1d2-mcp-server/agent.py", "serve"],
    )
    # Jupyter 中 sys.stderr 是 ipykernel OutStream，没有 fileno()
    # subprocess.Popen 需要有底层文件描述符的 stderr，所以用 os.devnull 替代
    with open(_os.devnull, "w") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()   # MCP 握手：交换协议版本和能力
                return await run_agent(session, llm_client, user_query, verbose=verbose)

print("ask() 助手已就绪！用法: await ask('你的问题')")

## 9. 交互测试

### 9.1 单次查询测试

修改下面 `query` 变量的值，然后运行这个 Cell。观察 Agent 通过 MCP 调用工具的完整链路。

**试试这些：**
- `"深圳今天天气怎么样？"`
- `"帮算一下 12345 * 67890"`
- `"计算 sin(pi/6) + cos(pi/3)"`

In [ ]:
# ===== 修改这行，测试不同的查询 =====
query = "北京今天天气怎么样？适合出去玩吗？"
# =====================================

print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

### 9.2 更多测试场景

逐一运行下面每个 Cell，观察 MCP Agent 在不同场景下的调用模式。
特别注意对比与昨天纯 Function Calling 的差异。

In [ ]:
# 场景 A: 单工具 — 复杂两步计算
query = "帮我算一下 2 的 20 次方是多少？然后再把结果除以 1024"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 B: 并行调用 — 一次查两个城市
query = "上海和广州现在的天气分别怎么样？哪个更适合出门？"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 C: 多步推理 — 先查天气，再根据结果计算
query = "北京现在多少度？如果北京比成都热 5 度，成都应该是多少度？"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 D: 英文输入 — 测试跨语言处理
query = "What's the weather in Tokyo and New York? Answer in Chinese please."
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent:\n{answer}")

In [ ]:
# 场景 E: 几何计算 — 三角函数
query = "一个角度为30度的直角三角形，斜边长为10，请帮我计算两条直角边的长度"
print(f"用户: {query}")
answer = await ask(query, verbose=True)
print(f"\nAgent:\n{answer}")

### 9.3 自由输入

输入你自己的问题，看看 MCP Agent 如何应对。

**挑战题目：**
- 问一个和天气、计算都无关的问题 → Agent 会乱调工具吗？
- 问"帮我算 3+5，顺便查一下北京天气" → 会并行调用吗？
- 对比昨天：同样的问题，MCP 版的调用链路有何不同？

In [ ]:
# ===== 自由发挥区：输入任何你想问的 =====
query = input("请输入你的问题: ").strip()
if not query:
    query = "今天深圳天气如何？帮我算一下 1024 * 768 的结果"
    print(f"(使用默认问题): {query}")

print()
answer = await ask(query, verbose=True)
print(f"\nAgent 最终回复:\n{answer}")

## 10. 深入理解：消息历史对比

运行下面代码，查看完整的 MCP 调用链路产生的消息历史。
特别注意：MCP 版本的 tool 消息内容与昨天纯 FC 版本完全一致——
这说明 MCP 只是改变了**调用通道**，数据格式不变。

In [ ]:
import os as _os

async def run_agent_with_history(
    llm_client: AsyncOpenAI,
    user_query: str,
    model: str = "deepseek-v4-flash",
    max_turns: int = 10,
) -> tuple[str, list]:
    """带 MCP 会话的完整历史记录版 Agent——每次自动创建连接并返回消息历史"""
    server_params = StdioServerParameters(
        command="uv",
        args=["run", "python", "exercises/w1d2-mcp-server/agent.py", "serve"],
    )
    with open(_os.devnull, "w") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                tools_result = await session.list_tools()
                openai_tools = mcp_tools_to_openai(tools_result.tools)

                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_query},
                ]

                for turn in range(1, max_turns + 1):
                    response = await llm_client.chat.completions.create(
                        model=model, messages=messages, tools=openai_tools,
                        tool_choice="auto", temperature=0.0,
                    )
                    msg = response.choices[0].message

                    if msg.tool_calls:
                        messages.append(msg.model_dump())
                        for tc in msg.tool_calls:
                            args = json.loads(tc.function.arguments)
                            result = await session.call_tool(tc.function.name, args)
                            messages.append({
                                "role": "tool",
                                "tool_call_id": tc.id,
                                "content": result.content[0].text,
                            })
                    else:
                        return msg.content, messages

    return "超时", messages


# 执行并查看消息历史
final_answer, history = await run_agent_with_history(
    llm_client, "北京天气怎么样？"
)

print("=" * 60)
print("MCP Agent 完整消息历史 (共", len(history), "条)")
print("=" * 60)

for i, msg in enumerate(history):
    role = msg["role"]
    content = msg.get("content", "")

    if role == "tool":
        # MCP 版和纯 FC 版的 tool 消息格式完全一致——这证明 MCP 只改了调用通道
        print(f"\n[{i}] role=tool  tool_call_id={msg['tool_call_id'][:8]}...")
        print(f"    content: {content[:300]}")
    elif role == "assistant" and "tool_calls" in msg:
        for tc in msg["tool_calls"]:
            print(f"\n[{i}] role=assistant -> tool_call: {tc['function']['name']}")
            print(f"    arguments: {tc['function']['arguments']}")
    elif role == "assistant":
        print(f"\n[{i}] role=assistant (final)")
        print(f"    content: {content[:300]}")
    else:
        preview = (content or "")[:200].replace("\n", " ")
        print(f"\n[{i}] role={role}: {preview}...")

print(f"\n\n最终回复:\n{final_answer}")

## 11. 架构对比：MCP vs 纯 Function Calling

### 调用链路对比

```
纯 Function Calling（昨天）:
  user → Agent Loop → LLM
                    → tool_calls → execute_tool() [本地函数调用]
                    → 回传结果 → LLM → final_answer

MCP 架构（今天）:
  user → Agent Loop → LLM
                    → list_tools() [MCP: 发现工具]
                    → tool_calls → session.call_tool() [MCP: 远程调用]
                    → Server 执行 → 回传结果 → LLM → final_answer
```

### 优劣对比

| 维度 | 纯 Function Calling | MCP 架构 |
|------|-------------------|---------|
| **工具定义** | 硬编码在 Agent 代码中 | Server 统一管理 |
| **可复用性** | 每个项目重新定义 | 一次定义，多处使用 |
| **语言无关** | 同语言调用 | 任何语言实现 Server |
| **部署灵活性** | 与 Agent 同进程 | 可独立部署、独立扩缩 |
| **复杂度** | 低（适合简单场景） | 中（适合生产系统） |
| **调试难度** | 容易（本地调用） | 需理解进程通信 |

> **选择建议**: 原型验证用纯 FC，生产系统用 MCP。今天的学习让你可以两条路都走。

## 12. 参考资料

| 主题 | 链接 | 说明 |
|------|------|------|
| MCP 官方文档 | https://modelcontextprotocol.io | MCP 协议总览 |
| MCP 规范 | https://spec.modelcontextprotocol.io | 协议详细规范 |
| MCP Python SDK | https://github.com/modelcontextprotocol/python-sdk | Python 实现 |
| OpenAI Function Calling | https://platform.openai.com/docs/guides/function-calling | FC 官方文档 |
| DeepSeek API | https://api-docs.deepseek.com/zh-cn/ | DeepSeek API 文档 |
| Anthropic Tool Use | https://docs.anthropic.com/en/docs/build-with-claude/tool-use | Claude 工具使用 |

### 明日预告（2026-05-08）

给 Agent 加错误重试 + 工具结果验证！

参考: Anthropic《Writing Effective Tools》: https://www.anthropic.com/engineering/writing-tools-for-agents

内容：工具调用失败怎么办？超时重试、结果校验、降级策略——把 Agent 从 Demo 级别提升到生产级别。

## 13. 核心知识点回顾

### 今天你掌握了什么

| # | 知识点 | 对应位置 |
|---|--------|---------|
| 1 | **MCP 协议三大原语** — Tools / Resources / Prompts | Cell 2 |
| 2 | **MCP Server 实现** — `list_tools()` + `call_tool()` 两个核心方法 | Cell 4 |
| 3 | **MCP → OpenAI 格式转换** — `inputSchema` 即 `parameters` | Cell 5 |
| 4 | **MCP Client 交互** — stdio 启动 Server → initialize → list_tools → call_tool | Cell 8 |
| 5 | **Agent 循环 MCP 版** — 工具发现和执行都通过 MCP Session | Cell 7 |
| 6 | **MCP vs FC 对比** — MCP 管供应链，FC 管消费端 | Cell 2 & 11 |

### 对应八股题

| 题号 | 主题 | 对应 Cell |
|------|------|-----------|
| 题 12 | MCP 协议核心概念 | Cell 2 (理论) + Cell 4 (实现) |
| 题 13 | MCP 与 Function Calling 关系 | Cell 2 (对比) + Cell 5 (转换) |

### 代码文件

- `exercises/w1d2-mcp-server/agent.py` — 完整可运行脚本（Server + Client + 6 测试用例）
- `mcp_agent.ipynb` — 交互式学习 Notebook（当前文件）
- `function_calling_agent.py` — 昨天的纯 FC 版本（对比参考）
- `function_calling_agent.ipynb` — 昨天的纯 FC Notebook（对比参考）

---

> MCP 是 Agent 工具系统的"USB-C 接口"——标准化带来生态。
> 理解了 MCP，你就能让任何 Agent 调用任何工具，不受语言和框架的限制。
> 明天我们继续强化 Agent 的健壮性——加上错误处理和重试逻辑。

## 学习检查清单

读完这节后，建议你能回答：

- MCP Server 暴露的工具，为什么还要转换成 Function Calling schema？
- `stdio_client` 在这个 demo 里解决了什么连接问题？
- MCP 相比直接本地函数调用，主要工程价值是什么？
- 如果以后工具由另一个团队维护，MCP 会如何降低耦合？
- 为什么 notebook/Jupyter 环境下 subprocess 的 stderr/stdin/stdout 更容易出坑？
